In [1]:
import pandas as pd

# 1. Загружаем исходный CSV
CSV_PATH = "../data/raw/yellow_tripdata_2015-01.csv"
df_raw = pd.read_csv(CSV_PATH)

# 2. Оцениваем объём памяти по каждому столбцу (deep=True честно считает строки в байтах)
print("ОБЪЁМ ПАМЯТИ ПО СТОЛБЦАМ (в КБ) ")
memory_per_col = df_raw.memory_usage(deep=True) / 1024
print(memory_per_col.round(2))

total_memory_mb = memory_per_col.sum() / 1024
print(f"\nОбщий объём памяти в RAM до оптимизации: {total_memory_mb:.2f} МБ\n")

# 3. Вычисляем долю уникальных значений для текстовых столбцов
print("ДОЛЯ УНИКАЛЬНЫХ ЗНАЧЕНИЙ В ТЕКСТОВЫХ СТОЛБЦАХ ")
string_cols = df_raw.select_dtypes(include=["object"]).columns

for col in string_cols:
    unique_count = df_raw[col].nunique()
    total_count = len(df_raw)
    ratio = unique_count / total_count
    print(
        f"Столбец '{col}': {unique_count} уникальных из {total_count} ({ratio:.4%})"
    )

ОБЪЁМ ПАМЯТИ ПО СТОЛБЦАМ (в КБ) 
Index                         0.13
VendorID                  99601.45
tpep_pickup_datetime     846612.35
tpep_dropoff_datetime    846612.35
passenger_count           99601.45
trip_distance             99601.45
pickup_longitude          99601.45
pickup_latitude           99601.45
RateCodeID                99601.45
store_and_fwd_flag       622509.08
dropoff_longitude         99601.45
dropoff_latitude          99601.45
payment_type              99601.45
fare_amount               99601.45
extra                     99601.45
mta_tax                   99601.45
tip_amount                99601.45
tolls_amount              99601.45
improvement_surcharge     99601.45
total_amount              99601.45
dtype: float64

Общий объём памяти в RAM до оптимизации: 3817.73 МБ

ДОЛЯ УНИКАЛЬНЫХ ЗНАЧЕНИЙ В ТЕКСТОВЫХ СТОЛБЦАХ 
Столбец 'tpep_pickup_datetime': 2438284 уникальных из 12748986 (19.1253%)
Столбец 'tpep_dropoff_datetime': 2442163 уникальных из 12748986 (19.1557%)
Ст

In [2]:
import sys

# Добавляем корень проекта в путь импорта (если вызываем из папки notebooks/)
sys.path.append("..")

import pandas as pd

from src.config import DATE_COLUMNS, TAXI_DTYPES

# Загружаем данные сразу с использованием нашей схемы
df_opt = pd.read_csv(
    CSV_PATH,
    dtype=TAXI_DTYPES,
    parse_dates=DATE_COLUMNS,
)

# Проверяем новый объём памяти
total_memory_opt_mb = df_opt.memory_usage(deep=True).sum() / (1024 * 1024)

print(f"Объём памяти ДО оптимизации:   {total_memory_mb:.2f} МБ")
print(f"Объём памяти ПОСЛЕ оптимизации: {total_memory_opt_mb:.2f} МБ")
print(
    f"Экономия памяти: {((total_memory_mb - total_memory_opt_mb) / total_memory_mb) * 100:.2f}%"
)

# Проверяем типы столбцов
print("\nИтоговые типы данных:")
print(df_opt.dtypes)

Объём памяти ДО оптимизации:   3817.73 МБ
Объём памяти ПОСЛЕ оптимизации: 838.93 МБ
Экономия памяти: 78.03%

Итоговые типы данных:
VendorID                       category
tpep_pickup_datetime     datetime64[ns]
tpep_dropoff_datetime    datetime64[ns]
passenger_count                   uint8
trip_distance                   float32
pickup_longitude                float32
pickup_latitude                 float32
RateCodeID                     category
store_and_fwd_flag             category
dropoff_longitude               float32
dropoff_latitude                float32
payment_type                   category
fare_amount                     float32
extra                           float32
mta_tax                         float32
tip_amount                      float32
tolls_amount                    float32
improvement_surcharge           float32
total_amount                    float32
dtype: object


In [3]:
import os
import statistics
import sys
import time
from pathlib import Path

# Добавляем корень проекта для импорта src.config
sys.path.append("..")

import pandas as pd

from src.config import DATE_COLUMNS, TAXI_DTYPES

# Пути к файлам
CSV_PATH = Path("../data/raw/yellow_tripdata_2015-01.csv")
PARQUET_BEFORE = Path("../data/interim/data_before_opt.parquet")
PARQUET_AFTER = Path("../data/interim/data_after_opt.parquet")


# Вспомогательная функция для замеров времени (выбор медианы из 3 запусков)
def measure_median_time(action_fn, runs=3):
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        action_fn()
        end = time.perf_counter()
        times.append(end - start)
    return statistics.median(times)


# 1. ЗАМЕРЫ ДО ОПТИМИЗАЦИИ 
# Время чтения исходного CSV
t_read_before = measure_median_time(lambda: pd.read_csv(CSV_PATH))

df_before = pd.read_csv(CSV_PATH)
df_before.to_parquet(PARQUET_BEFORE)

# Размер на диске и в RAM
size_disk_before = os.path.getsize(PARQUET_BEFORE) / (1024 * 1024)
ram_before = df_before.memory_usage(deep=True).sum() / (1024 * 1024)

# Время группировки
t_groupby_before = measure_median_time(
    lambda: df_before.groupby("payment_type")["total_amount"].mean()
)
cat_cols_before = len(df_before.select_dtypes(include=["category"]).columns)


#2. ЗАМЕРЫ ПОСЛЕ ОПТИМИЗАЦИИ 
# Время чтения с использованием разработанной схемы
t_read_after = measure_median_time(
    lambda: pd.read_csv(CSV_PATH, dtype=TAXI_DTYPES, parse_dates=DATE_COLUMNS)
)

df_after = pd.read_csv(
    CSV_PATH, dtype=TAXI_DTYPES, parse_dates=DATE_COLUMNS
)
df_after.to_parquet(PARQUET_AFTER)

# Размер на диске и в RAM
size_disk_after = os.path.getsize(PARQUET_AFTER) / (1024 * 1024)
ram_after = df_after.memory_usage(deep=True).sum() / (1024 * 1024)

# Время группировки
t_groupby_after = measure_median_time(
    lambda: df_after.groupby("payment_type")["total_amount"].mean()
)
cat_cols_after = len(df_after.select_dtypes(include=["category"]).columns)


# 3. ФОРМИРОВАНИЕ И ВЫВОД ТАБЛИЦЫ 7 
def calc_diff(before, after, is_time=False):
    pct = ((before - after) / before) * 100
    return f"-{pct:.1f}%" if pct > 0 else f"+{abs(pct):.1f}%"


table7_data = [
    {
        "Показатель": "Объём файла на диске, МБ",
        "До оптимизации": f"{size_disk_before:.2f}",
        "После": f"{size_disk_after:.2f}",
        "Изменение": calc_diff(size_disk_before, size_disk_after),
    },
    {
        "Показатель": "Объём в памяти, МБ",
        "До оптимизации": f"{ram_before:.2f}",
        "После": f"{ram_after:.2f}",
        "Изменение": calc_diff(ram_before, ram_after),
    },
    {
        "Показатель": "Время чтения, с",
        "До оптимизации": f"{t_read_before:.4f}",
        "После": f"{t_read_after:.4f}",
        "Изменение": calc_diff(t_read_before, t_read_after),
    },
    {
        "Показатель": "Время группировки, с",
        "До оптимизации": f"{t_groupby_before:.4f}",
        "После": f"{t_groupby_after:.4f}",
        "Изменение": calc_diff(t_groupby_before, t_groupby_after),
    },
    {
        "Показатель": "Число столбцов типа category",
        "До оптимизации": str(cat_cols_before),
        "После": str(cat_cols_after),
        "Изменение": f"+{cat_cols_after - cat_cols_before}",
    },
]

df_table7 = pd.DataFrame(table7_data)
print("\n Таблица 7 - Результаты оптимизации схемы данных ")
print(df_table7.to_string(index=False))

C:\Users\cendv\AppData\Local\Temp\ipykernel_5236\3552325938.py:66: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  lambda: df_after.groupby("payment_type")["total_amount"].mean()



 Таблица 7 - Результаты оптимизации схемы данных 
                  Показатель До оптимизации   После Изменение
    Объём файла на диске, МБ         305.03  311.01     +2.0%
          Объём в памяти, МБ        3817.73  838.93    -78.0%
             Время чтения, с        41.1506 98.8834   +140.3%
        Время группировки, с         0.2639  0.1619    -38.7%
Число столбцов типа category              0       4        +4
